## Project Requirements (reference)

Your datasets should be suitable for logistic regression and SVMs. That is, they should have at least one variable that could be a target categorical variable. The remaining variables you will consider for explanatory variables can be categorical or quantitative, but your analysis might be a little easier if you choose quantitative variables. The target variable must have at least two classes, but be careful that it doesn't have too many classes or your predictive and runtime performance might suffer. Questions you could address in your report include the following. (You should talk about at least four of them across all datasets -- each individual dataset doesn't need four questions addressed.)

- Which variables are predictive of the target variable?
- Can logistic regression or a linear SVM predict well?
- What do plots of selected pairs of variables look like? Where is the decision boundary in those plots?
- How generalizable are the different models on your data? How does the bias-variance tradeoff affect which model you might choose?
- Is there a difference between the polynomial and RBF SVMs?
- What effect does changing the class_weight in an SVM have on your data? How might this be important for this data?
- Is there a difference in runtime performance?
- Logistic regression and LinearSVC use one-vs-rest (OVR) for multi-class classification. SVC uses one-vs-one (OVO). Where n is the number of classes, OVR learns n models, whereas OVO learns n(n-1)/2 (n choose 2) models. What effect does this have on performance?

# Dataset 2: Breast Cancer Classification with Logistic Regression and SVMs

## Research question
How well can logistic regression and support vector machines distinguish malignant from benign tumors using the numeric measurements in scikit-learn's Wisconsin Diagnostic Breast Cancer dataset? Because missed malignant cases matter, malignant is explicitly treated as the positive class and recall is a primary comparison metric. This is a classroom analysis of a benchmark dataset, not a clinical diagnostic tool.

The built-in dataset contains 569 cases, 30 numeric features, and two outcome classes (212 malignant and 357 benign). The measurements summarize cell nuclei from digitized fine needle aspirate images.

## Class-example connection
The course Logistic Regression notebook (`examples/l09-logistic-regression-20261108T215809Z-1-001/l09-logistic-regression/logistic-regression.ipynb`) motivates probabilities, the default 0.5 cutoff, precision/recall/F-score, and a linear decision boundary. The SVM notebook (`examples/l10-support-vector-machines-20261108T215813Z-1-001/l10-support-vector-machines/support-vector-machines.ipynb`) compares linear, polynomial, and RBF kernels, illustrates decision boundaries, and explores class weights. This analysis follows those ideas while using a held-out test set and training-only cross-validation for evaluation.

Questions addressed here: which features appear predictive; whether linear and kernel classifiers perform well; how polynomial and RBF SVMs compare; how class weighting changes the recall/precision tradeoff; and how the fitted decision boundaries look for a selected feature pair.

## 1. Data, target coding, and quality checks

The target is recoded from scikit-learn's encoded labels to `1 = malignant` and `0 = benign`, so recall and sensitivity refer to malignant cases throughout. Feature preparation and scaling will be fitted inside each model pipeline to prevent validation or test information from influencing training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score, roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print("Imports and reproducibility settings ready.")

In [ ]:
cancer = load_breast_cancer(as_frame=True)
X = cancer.data.copy()
malignant_code = list(cancer.target_names).index("malignant")
y = (cancer.target == malignant_code).astype(int).rename("malignant")

print(f"Shape: {X.shape[0]} cases x {X.shape[1]} features")
print(f"Original target encoding: {dict(enumerate(cancer.target_names))}")
print("Analysis coding: 1 = malignant; 0 = benign")
print(f"Missing cells: {int(X.isna().sum().sum())}; exact duplicate feature rows: {int(X.duplicated().sum())}")
display(pd.DataFrame({"count": y.value_counts().sort_index(), "class": ["benign", "malignant"]}))
display(X.describe().T[["mean", "std", "min", "max"]].round(3).head(10))

There are no missing feature values. Duplicate feature rows are checked and reported rather than removed automatically: without case identifiers or a documented repeated-measure design, dropping observations solely because their measurements match would need justification. The next plots summarize the class balance and selected feature distributions.

In [ ]:
class_counts = y.map({0: "Benign", 1: "Malignant"}).value_counts().reindex(["Benign", "Malignant"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(class_counts.index, class_counts.values, color=["#0072B2", "#D55E00"])
axes[0].set(title="Outcome class counts", ylabel="Cases")
axes[0].bar_label(axes[0].containers[0])

for ax, feature in zip(axes[1:], ["mean radius"]):
    for label, color in [(0, "#0072B2"), (1, "#D55E00")]:
        ax.hist(X.loc[y.eq(label), feature], bins=22, alpha=0.6, density=True,
                color=color, label={0: "Benign", 1: "Malignant"}[label])
    ax.set(title=feature, xlabel=feature, ylabel="Density")
    ax.legend()
fig.tight_layout()
plt.show()

## 2. Exploratory analysis

Feature units and spreads differ, so standardization is important for the SVM distance calculations and regularized logistic regression. Several features are also strongly correlated, which means individual model coefficients should not be read as independent or causal effects.

In [ ]:
selected_features = ["mean radius", "mean texture", "mean concave points", "worst perimeter"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, feature in zip(axes.flat, selected_features):
    for label, color in [(0, "#0072B2"), (1, "#D55E00")]:
        ax.hist(X.loc[y.eq(label), feature], bins=22, alpha=0.55, density=True,
                color=color, label={0: "Benign", 1: "Malignant"}[label])
    ax.set(title=feature, xlabel=feature, ylabel="Density")
axes[0, 0].legend()
fig.suptitle("Selected measurements by diagnosis", y=1.02, fontweight="bold")
fig.tight_layout()
plt.show()

corr = X.corr()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
top_correlations = upper.reindex(upper.abs().sort_values(ascending=False).head(10).index)
display(top_correlations.rename("Pearson correlation").to_frame().round(3))

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr.columns)), corr.columns, fontsize=6)
ax.set_title("Correlations among the 30 features")
fig.colorbar(image, ax=ax, shrink=0.75, label="Pearson r")
fig.tight_layout()
plt.show()

## 3. Train/test protocol and model tuning

A stratified 80/20 split reserves one fifth of the observations for final evaluation. Five-fold stratified cross-validation on the training partition selects hyperparameters. The primary tuning score is malignant recall because missing malignant cases is the central error of interest; precision and balanced accuracy are also tracked so that a recall gain is not mistaken for an unqualified improvement. The default prediction threshold is retained (0.5 probability for logistic regression; zero decision score for SVMs).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
scoring = {"recall": "recall", "precision": "precision", "balanced_accuracy": "balanced_accuracy"}

specifications = {
    "Logistic regression": (
        LogisticRegression(max_iter=5000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1, 10, 100], "model__class_weight": [None, "balanced"]},
    ),
    "Linear SVM": (
        LinearSVC(dual="auto", max_iter=20000, random_state=RANDOM_STATE),
        {"model__C": [0.01, 0.1, 1, 10, 100], "model__class_weight": [None, "balanced"]},
    ),
    "Polynomial SVM": (
        SVC(kernel="poly", cache_size=500),
        {"model__C": [0.1, 1, 10], "model__degree": [2, 3],
         "model__class_weight": [None, "balanced"]},
    ),
    "RBF SVM": (
        SVC(kernel="rbf", cache_size=500),
        {"model__C": [0.1, 1, 10, 100], "model__gamma": ["scale", 0.01, 0.1],
         "model__class_weight": [None, "balanced"]},
    ),
}

searches = {}
for name, (estimator, param_grid) in specifications.items():
    pipeline = Pipeline([ ("scale", StandardScaler()), ("model", estimator) ])
    search = GridSearchCV(
        pipeline, param_grid, scoring=scoring, refit="recall", cv=cv,
        n_jobs=-1, return_train_score=True,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name}: CV malignant recall={search.best_score_:.3f}; best params={search.best_params_}")

The baseline and four tuned classifiers are now compared on the same held-out observations. ROC-AUC uses each model's continuous decision score (or probability for logistic regression), while the class metrics use each estimator's standard prediction rule.

In [ ]:
def classification_metrics(model, X_eval, y_eval):
    prediction = model.predict(X_eval)
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_eval)[:, 1]
    else:
        score = model.decision_function(X_eval)
    return {
        "Accuracy": accuracy_score(y_eval, prediction),
        "Balanced accuracy": balanced_accuracy_score(y_eval, prediction),
        "Malignant recall / sensitivity": recall_score(y_eval, prediction, pos_label=1),
        "Specificity": recall_score(y_eval, prediction, pos_label=0),
        "Malignant precision": precision_score(y_eval, prediction, pos_label=1, zero_division=0),
        "Malignant F1": f1_score(y_eval, prediction, pos_label=1),
        "ROC-AUC": roc_auc_score(y_eval, score),
        "Average precision": average_precision_score(y_eval, score),
    }

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
fitted_models = {"Majority baseline": dummy}
fitted_models.update({name: search.best_estimator_ for name, search in searches.items()})
test_results = pd.DataFrame({name: classification_metrics(model, X_test, y_test)
                            for name, model in fitted_models.items()}).T
display(test_results.sort_values("Malignant recall / sensitivity", ascending=False).round(3))

## 4. How class weighting changes the SVM tradeoff

As in the class SVM example, compare unweighted and balanced class penalties. For each setting, select `C` using training-fold malignant recall, then compare the resulting models on the same test set. The minority class is malignant in this dataset, but the class imbalance is moderate; weighting is assessed empirically rather than assumed to help.

In [ ]:
linear_search = searches["Linear SVM"]
cv_rows = pd.DataFrame(linear_search.cv_results_)
weight_models = {}
weight_rows = []
weight_column = cv_rows["param_model__class_weight"]
for weight in [None, "balanced"]:
    if weight is None:
        weight_mask = weight_column.isna() | weight_column.map(lambda value: value is None)
    else:
        weight_mask = weight_column.eq(weight).fillna(False)
    subset = cv_rows.loc[weight_mask]
    best_row = subset.loc[subset["mean_test_recall"].idxmax()]
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", LinearSVC(C=float(best_row["param_model__C"]), class_weight=weight,
                             dual="auto", max_iter=20000, random_state=RANDOM_STATE)),
    ])
    model.fit(X_train, y_train)
    weight_models[str(weight)] = model
    row = {"SVM class_weight": str(weight), "Selected C": float(best_row["param_model__C"]),
           "CV recall": best_row["mean_test_recall"],
           "CV precision": best_row["mean_test_precision"],
           "CV balanced accuracy": best_row["mean_test_balanced_accuracy"]}
    row.update({f"Test {key}": value for key, value in classification_metrics(model, X_test, y_test).items()
                if key in ["Malignant recall / sensitivity", "Specificity", "Malignant precision", "Balanced accuracy"]})
    weight_rows.append(row)

weight_comparison = pd.DataFrame(weight_rows).set_index("SVM class_weight")
display(weight_comparison.round(3))

A rise in malignant recall can be useful, but it may also label more benign cases as malignant, lowering precision or specificity. The table reports both sides of that tradeoff. It does not declare either weighting choice universally best.

## 5. Confusion matrices and ROC curves

These displays compare the models on the held-out test partition. In every confusion matrix, the positive class is malignant.

In [ ]:
model_items = list(fitted_models.items())
fig, axes = plt.subplots(1, len(model_items), figsize=(4 * len(model_items), 3.5))
for ax, (name, model) in zip(axes, model_items):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test, display_labels=["Benign", "Malignant"],
        cmap="Blues", colorbar=False, ax=ax, values_format="d",
    )
    ax.set_title(name, fontsize=9)
fig.suptitle("Held-out confusion matrices", y=1.04, fontweight="bold")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, model in list(fitted_models.items())[1:]:
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_test)[:, 1]
    else:
        score = model.decision_function(X_test)
    RocCurveDisplay.from_predictions(y_test, score, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_title("Held-out ROC curves (positive = malignant)")
plt.show()

## 6. Feature importance and a two-feature boundary

Standardized logistic coefficients show the direction and relative size of the fitted linear association, conditional on all the other correlated features. Permutation importance gives a complementary, model-based ranking by measuring the change in held-out balanced accuracy when a feature is shuffled. Neither is evidence of causation. The two-feature decision boundaries below are trained only on the training split and are visual aids; they are not the fitted 30-feature decision surfaces.

In [ ]:
logistic = searches["Logistic regression"].best_estimator_.named_steps["model"]
coef = pd.Series(logistic.coef_[0], index=X.columns)
coef_top = coef.reindex(coef.abs().sort_values(ascending=False).head(12).index).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
coef_top.plot.barh(ax=ax, color=["#D55E00" if value > 0 else "#0072B2" for value in coef_top])
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title="Largest standardized logistic coefficients", xlabel="Coefficient (positive points toward malignant)")
fig.tight_layout()
plt.show()

best_name = max(searches, key=lambda name: searches[name].best_score_)
best_model = searches[best_name].best_estimator_
importance = permutation_importance(
    best_model, X_test, y_test, scoring="balanced_accuracy",
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1,
)
perm = pd.Series(importance.importances_mean, index=X.columns).sort_values().tail(12)
fig, ax = plt.subplots(figsize=(8, 5))
perm.plot.barh(xerr=pd.Series(importance.importances_std, index=X.columns).reindex(perm.index),
               ax=ax, color="#009E73", capsize=2)
ax.set(title=f"Held-out permutation importance: {best_name}", xlabel="Decrease in balanced accuracy after shuffling")
fig.tight_layout()
plt.show()
display(pd.DataFrame({"coefficient": coef, "permutation importance":
                      pd.Series(importance.importances_mean, index=X.columns)}).
       sort_values("permutation importance", ascending=False).head(10).round(4))

In [ ]:
pair = ["mean radius", "mean concave points"]
pair_models = {
    "Logistic regression": LogisticRegression(C=1, max_iter=5000, random_state=RANDOM_STATE),
    "RBF SVM": SVC(kernel="rbf", C=1, gamma="scale", cache_size=500),
}
x0, x1 = X_train[pair[0]], X_train[pair[1]]
xx, yy = np.meshgrid(np.linspace(x0.min(), x0.max(), 180), np.linspace(x1.min(), x1.max(), 180))
grid = pd.DataFrame({pair[0]: xx.ravel(), pair[1]: yy.ravel()})
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, estimator) in zip(axes, pair_models.items()):
    pair_pipeline = Pipeline([("scale", StandardScaler()), ("model", estimator)])
    pair_pipeline.fit(X_train[pair], y_train)
    region = np.asarray(pair_pipeline.predict(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, region, levels=[-0.5, 0.5, 1.5], colors=["#DDEBF7", "#FCE0D6"], alpha=0.85)
    ax.contour(xx, yy, region, levels=[0.5], colors="black", linewidths=1.2)
    for label, color, marker in [(0, "#0072B2", "o"), (1, "#D55E00", "^")]:
        mask = y_test.eq(label)
        ax.scatter(X_test.loc[mask, pair[0]], X_test.loc[mask, pair[1]],
                   s=18, alpha=0.7, c=color, marker=marker, label={0: "Benign", 1: "Malignant"}[label])
    ax.set(title=name, xlabel=pair[0], ylabel=pair[1])
    ax.legend(fontsize=8)
fig.suptitle("Illustrative two-feature decision boundaries", y=1.02, fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Findings and limitations

### Model comparison
On this fixed 114-case test split, the linear SVM had the strongest malignant recall (0.976), with 0.991 accuracy, 0.988 balanced accuracy, 1.000 specificity, and 1.000 malignant precision. Logistic regression achieved 0.952 malignant recall and 0.986 specificity. The RBF SVM achieved 0.929 recall and 1.000 specificity, while the polynomial SVM achieved 0.857 recall and 1.000 specificity. The majority-class baseline had 0.632 accuracy but zero malignant recall, showing why accuracy alone is misleading for this task.

Cross-validation selected the RBF SVM for the highest mean malignant recall (0.965), followed by logistic regression (0.959), the linear SVM (0.953), and polynomial SVM (0.929). The linear SVM's stronger score on the single held-out split than its CV ranking illustrates ordinary sampling variability; the test set is only one estimate, not proof that it will always be the best model. The nonlinear SVMs did not improve the held-out sensitivity over the linear SVM in this run.

### Class-weight comparison
For the linear SVM, using `class_weight="balanced"` raised training-CV malignant recall from 0.947 to 0.953. On the held-out cases, recall rose from 0.929 to 0.976 while specificity rose from 0.986 to 1.000 and precision from 0.975 to 1.000. On this split, balancing improved all three reported quantities; this is a result of the selected hyperparameters and these cases, not a guarantee that class weighting always helps.

### Features and interpretation
For the RBF SVM selected by cross-validation, held-out permutation importance ranked `worst texture`, `worst concavity`, and `mean concavity` highest by decrease in balanced accuracy. The logistic coefficient plot gives a complementary linear-model view. Several measurements are correlated, so coefficients and permutation importance may distribute or share predictive signal; these rankings are associations useful for this model comparison, not causal findings.

### Limitations
The dataset is small and comes from a single diagnostic study. The train/test split and five-fold cross-validation provide a basic generalization check, but this analysis does not report uncertainty intervals or external validation. Performance on this benchmark does not establish clinical validity, and no model here should be used for patient care.

## References
- Scikit-learn dataset documentation: <https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html>
- Wisconsin Diagnostic Breast Cancer dataset at UCI: <https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic>
- Class examples: `examples/l09-logistic-regression-20261108T215809Z-1-001/l09-logistic-regression/logistic-regression.ipynb` and `examples/l10-support-vector-machines-20261108T215813Z-1-001/l10-support-vector-machines/support-vector-machines.ipynb`.